In [ ]:
import pandas as pd

df=pd.read_csv("UCI_Credit_Card.csv")

bill_cols=['BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6']
df['avg_bill']=df[bill_cols].mean(axis=1)
df['avg_utilization']=df['avg_bill']/df['LIMIT_BAL']

pay_cols=['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6',]
df['months_late']=(df[pay_cols] > 0).sum(axis=1)

def categorize_status(payment):
    if payment < 0:
        return "Paid in Full"
    elif payment == 0:
        return "Minimum Paid"
    else:
        return "Late"

df['Current_Status']=df['PAY_0'].apply(categorize_status)
print(df['Current_Status'].value_counts())
df[pay_cols]

In [ ]:
df['delay_factor'] = df['months_late']/6
df['util_factor'] = df['avg_utilization'].clip(0,1)
df['Risk_score'] = df['delay_factor']*70 + df['util_factor']*30

def get_risk_tier (score):
    if score <= 10: return "Very Low"
    elif score <= 30: return "Low"
    elif score <= 60: return "Medium"
    elif score <=85: return "High"
    else: return "Critical"

df['Risk_tier'] = df['Risk_score'].apply(get_risk_tier)
df.head()


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,PAY_AMT6,default.payment.next.month,avg_bill,avg_utilization,months_late,Current_Status,delay_factor,util_factor,Risk_score,Risk_tier
0,1,20000.0,2,2,1,24,2,2,-1,-1,...,0.0,1,1284.000000,0.064200,2,Late,0.333333,0.064200,25.259333,Low
1,2,120000.0,2,2,2,26,-1,2,0,0,...,2000.0,1,2846.166667,0.023718,2,Paid in Full,0.333333,0.023718,24.044875,Low
2,3,90000.0,2,2,2,34,0,0,0,0,...,5000.0,0,16942.166667,0.188246,0,Minimum Paid,0.000000,0.188246,5.647389,Very Low
3,4,50000.0,2,2,1,37,0,0,0,0,...,1000.0,0,38555.666667,0.771113,0,Minimum Paid,0.000000,0.771113,23.133400,Low
4,5,50000.0,1,2,1,57,-1,0,-1,0,...,679.0,0,18223.166667,0.364463,0,Paid in Full,0.000000,0.364463,10.933900,Low


In [29]:
# Group by your new Risk Tier and calculate the actual default rate
validation = df.groupby('Risk_tier')['default.payment.next.month'].mean().sort_values()

print("Actual Default Rate by Tier:")
print(validation)

Actual Default Rate by Tier:
Risk_tier
Very Low    0.112869
Low         0.174942
Medium      0.425477
High        0.635462
Critical    0.669110
Name: default.payment.next.month, dtype: float64


In [36]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

In [ ]:
features=['LIMIT_BAL', 'PAY_0', 'avg_bill', 'avg_utilization', 'months_late']
X=df[features]
Y=df['default.payment.next.month']

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size =0.2, random_state = 42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model=LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train_scaled, Y_train)


Y_pred=model.predict(X_test_scaled)
print("model accuracy profile")
print(classification_report(Y_test, Y_pred))
print(f"AUC Score: {roc_auc_score(Y_test, model.predict_proba(X_test_scaled)[:, 1]):.2f}")

model accuracy profile
              precision    recall  f1-score   support

           0       0.87      0.80      0.84      4687
           1       0.46      0.59      0.51      1313

    accuracy                           0.76      6000
   macro avg       0.66      0.70      0.68      6000
weighted avg       0.78      0.76      0.77      6000

AUC Score: 0.74


c:\Users\lukaj\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


In [44]:
ftr = ['LIMIT_BAL', 'PAY_0', 'avg_bill', 'avg_utilization', 'months_late', 'EDUCATION', 'PAY_AMT1']
X = df[ftr]
y = df['default.payment.next.month']

# 2. Split (We can reuse our previous X_train/X_test logic)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Initialize Random Forest 
# We use 'balanced' again to make sure we catch those defaults!
rf_model = RandomForestClassifier(n_estimators=200, max_depth=10, class_weight='balanced', random_state=42)

rf_model.fit(X_train, y_train)
rf_probs = rf_model.predict_proba(X_test)[:, 1]
rf_preds = rf_model.predict(X_test)

# 5. Results
print("--- Random Forest Performance ---")
print(f"New AUC Score: {roc_auc_score(y_test, rf_probs):.4f}")
print(classification_report(y_test, rf_preds))


--- Random Forest Performance ---
New AUC Score: 0.7746
              precision    recall  f1-score   support

           0       0.88      0.82      0.85      4687
           1       0.48      0.59      0.53      1313

    accuracy                           0.77      6000
   macro avg       0.68      0.71      0.69      6000
weighted avg       0.79      0.77      0.78      6000



In [39]:
rf_model.feature_importances_

array([0.09707449, 0.26848055, 0.11375677, 0.12086459, 0.26828803,
       0.0248404 , 0.10669518])

In [45]:
# 1. Get probabilities for both sets
train_probs = rf_model.predict_proba(X_train)[:, 1]
test_probs = rf_model.predict_proba(X_test)[:, 1]

# 2. Calculate AUC for both
train_auc = roc_auc_score(y_train, train_probs)
test_auc = roc_auc_score(y_test, test_probs)

print(f"Training AUC Score: {train_auc:.4f}")
print(f"Testing AUC Score:  {test_auc:.4f}")
print(f"The 'Generalization Gap': {train_auc - test_auc:.4f}")

Training AUC Score: 0.8562
Testing AUC Score:  0.7746
The 'Generalization Gap': 0.0816


In [48]:
import pickle

# 1. Save the Model (The 'Brain')
with open('credit_risk_model.pkl', 'wb') as f:
    pickle.dump(rf_model, f)

# 2. Attach the AI Probability to the WHOLE dataset
# This ensures every row in your SQL database has a 'Smart Score'
df['AI_Risk_Prob'] = rf_model.predict_proba(df[ftr])[:, 1]

# 3. Final Export for SQL
# We include the ID, the Features, the Target, and BOTH Risk Scores
final_columns = [
    'ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE',
    'avg_bill', 'avg_utilization', 'months_late', 
    'Risk_score', 'Risk_tier', 'AI_Risk_Prob', 'default.payment.next.month'
]

df[final_columns].to_csv('Credit_Data_Final_For_SQL.csv', index=False)
print("Project Data Exported Successfully!")

Project Data Exported Successfully!


In [49]:
import sqlite3
import pandas as pd

# 1. Load your final processed data
df_final = pd.read_csv('Credit_Data_Final_For_SQL.csv')

# 2. Connect to (or create) a local SQL database file
conn = sqlite3.connect('Bank_Credit_Risk.db')

# 3. Push the data into a table
# 'if_exists=replace' ensures that if you run this twice, it updates the data
df_final.to_sql('Credit_Risk_Analysis', conn, if_exists='replace', index=False)

print("SQL Database 'Bank_Credit_Risk.db' has been created with your scores!")
conn.close()

SQL Database 'Bank_Credit_Risk.db' has been created with your scores!


In [50]:
conn = sqlite3.connect('Bank_Credit_Risk.db')

query = """
SELECT 
    EDUCATION, 
    COUNT(*) as Total_Customers,
    AVG(AI_Risk_Prob) as Avg_AI_Risk,
    AVG(Risk_Score) as Avg_Manual_Risk
FROM Credit_Risk_Analysis
GROUP BY EDUCATION
ORDER BY Avg_AI_Risk DESC
"""

sql_results = pd.read_sql(query, conn)
print(sql_results)
conn.close()

   EDUCATION  Total_Customers  Avg_AI_Risk  Avg_Manual_Risk
0          3             4917     0.440902        24.587605
1          2            14030     0.423987        23.433599
2          1            10585     0.356702        15.960307
3          0               14     0.274035         7.025045
4          6               51     0.273963        15.903314
5          5              280     0.241294        16.313199
6          4              123     0.235379         9.147034


In [56]:
conn = sqlite3.connect('Bank_Credit_Risk.db')
cursor = conn.cursor()

# We use 'CREATE VIEW' so this logic is saved INSIDE the database file
cursor.execute("""
CREATE VIEW IF NOT EXISTS v_Executive_Dashboard AS
SELECT 
    ID,
    CASE 
        WHEN SEX = 1 THEN 'Male' 
        WHEN SEX = 2 THEN 'Female' 
        ELSE 'Other' 
    END as Gender,
    CASE 
        WHEN EDUCATION = 1 THEN 'Grad School'
        WHEN EDUCATION = 2 THEN 'University'
        WHEN EDUCATION = 3 THEN 'High School'
        ELSE 'Other'
    END as Education_Level,
    AGE,
    LIMIT_BAL as Credit_Limit,
    ROUND(AI_Risk_Prob * 100, 2) as AI_Risk_Percent,
    CASE 
        WHEN AI_Risk_Prob > 0.75 THEN 'Critical'
        WHEN AI_Risk_Prob > 0.50 THEN 'High'
        WHEN AI_Risk_Prob > 0.25 THEN 'Medium'
        ELSE 'Low'
    END as AI_Risk_Segment,
    Risk_Tier as Manual_Risk_Tier,
    CASE
        WHEN "default.payment.next.month" = 1 THEN 'Default'
        WHEN "default.payment.next.month" = 0 THEN 'Current'
    END as Payment_Status

FROM Credit_Risk_Analysis
""")

conn.commit()
print("SQL View 'v_Executive_Dashboard' created successfully!")
final_check = pd.read_sql("SELECT * FROM v_Executive_Dashboard LIMIT 5", conn)
print("--- SQL VIEW PREVIEW FOR POWER BI ---")
print(final_check)
conn.close()

SQL View 'v_Executive_Dashboard' created successfully!
--- SQL VIEW PREVIEW FOR POWER BI ---
   ID  Gender Education_Level  AGE  Credit_Limit  AI_Risk_Percent  \
0   1  Female      University   24       20000.0            82.19   
1   2  Female      University   26      120000.0            68.68   
2   3  Female      University   34       90000.0            26.08   
3   4  Female      University   37       50000.0            43.86   
4   5    Male      University   57       50000.0            33.34   

  AI_Risk_Segment Manual_Risk_Tier Payment_Status  
0        Critical              Low        Default  
1            High              Low        Default  
2          Medium         Very Low        Current  
3          Medium              Low        Current  
4          Medium              Low        Current  
